# Boston Dynamics — ML Model Training

This notebook trains predictive models for the Boston Dynamics Intelligence Agent:

1. **Predictive Maintenance** — Predicts component failure probability based on telemetry features
2. **Anomaly Detection** — Identifies abnormal telemetry patterns using statistical thresholds
3. **Remaining Useful Life (RUL)** — Estimates days until component replacement is needed

## Prerequisites
- BD_INTELLIGENCE database with populated tables
- Execute SQL scripts 01-04 before running this notebook
- Snowflake connection with appropriate permissions

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

from snowflake.snowpark.context import get_active_session
session = get_active_session()
print('Snowpark session ready')

In [ ]:
%%sql -r dataframe_1
USE DATABASE BD_INTELLIGENCE;
USE WAREHOUSE BOSTON_DYNAMICS_WH;

## 1. Load Training Data

In [ ]:
df_telemetry = session.sql("""
SELECT
    t.ROBOT_ID,
    r.ROBOT_MODEL,
    DATE_TRUNC('DAY', t.TIMESTAMP)::DATE AS READING_DATE,
    AVG(t.BATTERY_LEVEL) AS AVG_BATTERY,
    MIN(t.BATTERY_LEVEL) AS MIN_BATTERY,
    AVG(t.CPU_TEMPERATURE_C) AS AVG_CPU_TEMP,
    MAX(t.CPU_TEMPERATURE_C) AS MAX_CPU_TEMP,
    AVG(t.MOTOR_TEMPERATURE_C) AS AVG_MOTOR_TEMP,
    MAX(t.MOTOR_TEMPERATURE_C) AS MAX_MOTOR_TEMP,
    AVG(t.VIBRATION_MM_S) AS AVG_VIBRATION,
    MAX(t.VIBRATION_MM_S) AS MAX_VIBRATION,
    AVG(t.JOINT_LOAD_PERCENT) AS AVG_JOINT_LOAD,
    MAX(t.JOINT_LOAD_PERCENT) AS MAX_JOINT_LOAD,
    AVG(t.SPEED_M_S) AS AVG_SPEED,
    SUM(CASE WHEN t.ERROR_CODE IS NOT NULL THEN 1 ELSE 0 END) AS ERROR_COUNT,
    COUNT(*) AS READING_COUNT
FROM BD_INTELLIGENCE.RAW.ROBOT_TELEMETRY t
JOIN BD_INTELLIGENCE.RAW.ROBOTS r ON t.ROBOT_ID = r.ROBOT_ID
GROUP BY t.ROBOT_ID, r.ROBOT_MODEL, DATE_TRUNC('DAY', t.TIMESTAMP)::DATE
""").to_pandas()

df_components = session.sql("""
SELECT
    c.COMPONENT_ID,
    c.ROBOT_ID,
    r.ROBOT_MODEL,
    c.COMPONENT_TYPE,
    c.STATUS,
    DATEDIFF(DAY, c.INSTALL_DATE, CURRENT_DATE()) AS DAYS_SINCE_INSTALL,
    c.EXPECTED_LIFESPAN_DAYS,
    c.CURRENT_RUNTIME_HOURS,
    DATEDIFF(DAY, c.INSTALL_DATE, CURRENT_DATE()) * 1.0 / NULLIF(c.EXPECTED_LIFESPAN_DAYS, 0) AS LIFECYCLE_RATIO
FROM BD_INTELLIGENCE.RAW.COMPONENTS c
JOIN BD_INTELLIGENCE.RAW.ROBOTS r ON c.ROBOT_ID = r.ROBOT_ID
""").to_pandas()

print(f'Telemetry records: {len(df_telemetry)}')
print(f'Component records: {len(df_components)}')

## 2. Predictive Maintenance Model

Classifies components into risk categories (needs_attention vs operational) based on telemetry features and component lifecycle data.

In [ ]:
df_components['NEEDS_ATTENTION'] = df_components['STATUS'].isin(['Degraded', 'Needs Replacement', 'Failed']).astype(int)

le_model = LabelEncoder()
le_type = LabelEncoder()
df_components['MODEL_ENC'] = le_model.fit_transform(df_components['ROBOT_MODEL'])
df_components['TYPE_ENC'] = le_type.fit_transform(df_components['COMPONENT_TYPE'])

features = ['DAYS_SINCE_INSTALL', 'EXPECTED_LIFESPAN_DAYS', 'CURRENT_RUNTIME_HOURS',
            'LIFECYCLE_RATIO', 'MODEL_ENC', 'TYPE_ENC']
target = 'NEEDS_ATTENTION'

X = df_components[features].fillna(0)
y = df_components[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

maint_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
maint_model.fit(X_train, y_train)

y_pred = maint_model.predict(X_test)
print('=== Predictive Maintenance Model ===')
print(classification_report(y_test, y_pred, target_names=['Operational', 'Needs Attention']))

cv_scores = cross_val_score(maint_model, X, y, cv=5, scoring='accuracy')
print(f'Cross-validation accuracy: {cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})')

In [ ]:
importances = pd.DataFrame({
    'feature': features,
    'importance': maint_model.feature_importances_
}).sort_values('importance', ascending=False)

print('Feature Importances:')
print(importances.to_string(index=False))

## 3. Anomaly Detection

Uses statistical thresholds (Z-scores) on telemetry data to flag anomalous readings.

In [ ]:
anomaly_features = ['AVG_MOTOR_TEMP', 'MAX_MOTOR_TEMP', 'AVG_VIBRATION',
                    'MAX_VIBRATION', 'AVG_JOINT_LOAD', 'ERROR_COUNT']

df_anomaly = df_telemetry[anomaly_features].copy()

for col in anomaly_features:
    mean_val = df_anomaly[col].mean()
    std_val = df_anomaly[col].std()
    df_anomaly[f'{col}_ZSCORE'] = (df_anomaly[col] - mean_val) / std_val

zscore_cols = [c for c in df_anomaly.columns if 'ZSCORE' in c]
df_anomaly['MAX_ZSCORE'] = df_anomaly[zscore_cols].abs().max(axis=1)
df_anomaly['IS_ANOMALY'] = (df_anomaly['MAX_ZSCORE'] > 2.0).astype(int)

anomaly_count = df_anomaly['IS_ANOMALY'].sum()
total_count = len(df_anomaly)
print(f'=== Anomaly Detection Results ===')
print(f'Total readings analyzed: {total_count}')
print(f'Anomalies detected: {anomaly_count} ({anomaly_count/total_count*100:.1f}%)')
print(f'\nAnomaly distribution by Z-score threshold:')
for threshold in [2.0, 2.5, 3.0]:
    count = (df_anomaly['MAX_ZSCORE'] > threshold).sum()
    print(f'  Z > {threshold}: {count} readings ({count/total_count*100:.1f}%)')

## 4. Remaining Useful Life (RUL) Regression

Predicts estimated remaining days until a component needs replacement.

In [ ]:
df_rul = df_components.copy()
df_rul['REMAINING_DAYS'] = (df_rul['EXPECTED_LIFESPAN_DAYS'] - df_rul['DAYS_SINCE_INSTALL']).clip(lower=0)

rul_features = ['DAYS_SINCE_INSTALL', 'EXPECTED_LIFESPAN_DAYS', 'CURRENT_RUNTIME_HOURS',
                'LIFECYCLE_RATIO', 'MODEL_ENC', 'TYPE_ENC']

X_rul = df_rul[rul_features].fillna(0)
y_rul = df_rul['REMAINING_DAYS']

X_train_rul, X_test_rul, y_train_rul, y_test_rul = train_test_split(
    X_rul, y_rul, test_size=0.2, random_state=42
)

rul_model = GradientBoostingRegressor(n_estimators=100, max_depth=6, random_state=42)
rul_model.fit(X_train_rul, y_train_rul)

y_pred_rul = rul_model.predict(X_test_rul)

mae = mean_absolute_error(y_test_rul, y_pred_rul)
r2 = r2_score(y_test_rul, y_pred_rul)

print('=== Remaining Useful Life Model ===')
print(f'Mean Absolute Error: {mae:.1f} days')
print(f'R² Score: {r2:.3f}')
print(f'\nPrediction sample (first 10):')
comparison = pd.DataFrame({
    'Actual': y_test_rul.head(10).values,
    'Predicted': np.round(y_pred_rul[:10], 1)
})
print(comparison.to_string(index=False))

In [ ]:
rul_importances = pd.DataFrame({
    'feature': rul_features,
    'importance': rul_model.feature_importances_
}).sort_values('importance', ascending=False)

print('RUL Feature Importances:')
print(rul_importances.to_string(index=False))

## 5. Model Summary

In [ ]:
print('=' * 60)
print('BOSTON DYNAMICS ML MODEL TRAINING SUMMARY')
print('=' * 60)
print()
print('1. Predictive Maintenance (Random Forest Classifier)')
print(f'   - Cross-validation accuracy: {cv_scores.mean():.3f}')
print(f'   - Features: {len(features)}')
print(f'   - Training samples: {len(X_train)}')
print(f'   - Top feature: {importances.iloc[0]["feature"]}')
print()
print('2. Anomaly Detection (Z-Score Statistical)')
print(f'   - Threshold: Z > 2.0')
print(f'   - Anomaly rate: {anomaly_count/total_count*100:.1f}%')
print(f'   - Features monitored: {len(anomaly_features)}')
print()
print('3. Remaining Useful Life (Gradient Boosting Regressor)')
print(f'   - MAE: {mae:.1f} days')
print(f'   - R² Score: {r2:.3f}')
print(f'   - Training samples: {len(X_train_rul)}')
print(f'   - Top feature: {rul_importances.iloc[0]["feature"]}')
print()
print('Note: These models inform the SQL UDF functions in')
print('08_ml_model_functions.sql which are exposed to the agent.')
print('The UDFs use threshold-based logic derived from these models.')

print('\nNotebook complete.')